In [8]:
import sys
import os

# For notebooks, use getcwd(); for scripts, use __file__
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Running in a notebook
    notebook_dir = os.getcwd()

sys.path.insert(0, notebook_dir)

from agents.orchestrator import analyze_user_input, update_state_from_analysis
from agents.specialist import run_activities_specialist, run_logistics_specialist
from agents.verifier import verify_and_format_itinerary, format_complete_itinerary
from state import TravelState


In [2]:
state = TravelState()
task_json = {
  "task_id": "task_123",
  "intent": "update_constraints",
  "updated_constraints": {
    "origin": "BOS",
    "destination": "SEA",
    "start_date": "2026-05-05",
    "end_date": "2026-05-12"
  },
  "delegation": "logistics",
  "response_to_user": "I will search for flights...",
}
user_input = None #"want to change my trip to Seattle from May 6th to May 13th"

analysis = analyze_user_input(user_input, state, task_json=task_json)
state = update_state_from_analysis(state, analysis)
delegation = analysis.get("delegation", "none")

In [3]:
 # 3. Delegation
if delegation == "none":
    response = analysis.get(
        "response_to_user",
        "I'm still missing some info to start planning.",
    )
    print(f"\nNomad: {response}")
    state.messages.append({"role": "assistant", "content": response})

In [4]:

# If we reach here, we need to run specialists
draft_components = []
all_search_results = {
    "flights": [],
    "hotels": [],
    "activities": [],
}
# Accumulate specialist contexts for verifier
specialist_contexts = {
    "logistics": None,
    "activities": None,
}
constraints_str = state.constraints.model_dump_json(indent=2)

if delegation in ["logistics", "both"]:
    print("\n[Specialist - Logistics] Searching for flights & hotels...")
    try:
        # NEW: Unpack all 3 return values
        logistics_draft, logistics_searches, logistics_context = run_logistics_specialist(
            constraints_str, 
            task_id=state.task_id
        )
        draft_components.append("--- LOGISTICS ---\n" + logistics_draft)
        
        # Accumulate search results
        all_search_results["flights"].extend(logistics_searches.get("flights", []))
        all_search_results["hotels"].extend(logistics_searches.get("hotels", []))
        
        # Save specialist context for verifier
        specialist_contexts["logistics"] = logistics_context
        
        print(f"\n[Logistics Specialist Summary]")
        print(f"  Search Coverage: {logistics_context['search_coverage']}")
        print(f"  Total Searches: {logistics_context['total_searches']}")
        print(f"  Tool Calls: {len(logistics_context['tool_calls_summary'])}")
        
    except Exception as e:
        print(f"[Specialist Error] {e}")
        import traceback
        traceback.print_exc()



[Specialist - Logistics] Searching for flights & hotels...
  [Specialist Turn 1] Thinking...
  🔧 Executing tool search_flights with arguments {'origin': 'BOS', 'destination': 'SEA', 'departure_date': '2026-05-05', 'return_date': '2026-05-12', 'task_id': 'task_123', 'turn': 0}
[Cache Hit] google_flights
🍽️  Flights results: 3 options
[Saved] Candidate -> c:\GIT\aiagent\nomad\src\agents\..\search_candidates\task_123\flights_candidates.json (5 total)
  🔧 Executing tool search_hotels with arguments {'location': 'Seattle, Washington', 'check_in': '2026-05-05', 'check_out': '2026-05-12', 'adults': 1, 'task_id': 'task_123', 'turn': 0}
[Cache Hit] google_hotels
🍽️  Hotels results: 5 options
[Saved] Candidate -> c:\GIT\aiagent\nomad\src\agents\..\search_candidates\task_123\hotels_candidates.json (5 total)
  [Specialist Turn 2] Thinking...

[Logistics Specialist Summary]
  Search Coverage: {'flights': 6, 'hotels': 5, 'activities': 0}
  Total Searches: 11
  Tool Calls: 2


In [5]:
# View specialist context that will be passed to verifier
print("=" * 60)
print("SPECIALIST CONTEXT FOR VERIFIER")
print("=" * 60)

if specialist_contexts["logistics"]:
    ctx = specialist_contexts["logistics"]
    print(f"\n📊 LOGISTICS SPECIALIST ANALYSIS:")
    print(f"  • Search Coverage: {ctx['search_coverage']}")
    print(f"  • Total Search Results: {ctx['total_searches']}")
    print(f"  • Number of Tool Calls: {len(ctx['tool_calls_summary'])}")
    
    if ctx['tool_calls_summary']:
        print(f"\n  Tool Calls Summary:")
        for i, call in enumerate(ctx['tool_calls_summary'], 1):
            print(f"    {i}. {call['tool']} -> {call['result_count']} results (Turn {call['turn']})")
    
    if ctx['specialist_reasoning']:
        print(f"\n  Specialist Reasoning:")
        for reason in ctx['specialist_reasoning']:
            print(f"    → {reason[:100]}..." if len(reason) > 100 else f"    → {reason}")
    
    print(f"\n  Context Keys Available for Verifier:")
    for key in ctx.keys():
        print(f"    - {key}: {type(ctx[key]).__name__}")


SPECIALIST CONTEXT FOR VERIFIER

📊 LOGISTICS SPECIALIST ANALYSIS:
  • Search Coverage: {'flights': 6, 'hotels': 5, 'activities': 0}
  • Total Search Results: 11
  • Number of Tool Calls: 2

  Tool Calls Summary:
    1. search_flights -> 6 results (Turn 1)
    2. search_hotels -> 5 results (Turn 1)

  Specialist Reasoning:
    → I'll search for flights and hotels for your trip from Boston to Seattle from May 5-12, 2026. Let me ...
    → Perfect! I've found excellent flight and hotel options for your trip from Boston to Seattle from May...

  Context Keys Available for Verifier:
    - specialist_reasoning: list
    - search_coverage: dict
    - key_decisions: list
    - budget_analysis: str
    - risk_factors: list
    - recommendations: str
    - tool_calls_summary: list
    - final_draft: str
    - total_searches: int


In [ ]:

# Debug: Check accumulated search results before verifier
print("\n" + "=" * 60)
print("DEBUG: Search Results Accumulated")
print("=" * 60)
print(f"Flights: {len(all_search_results['flights'])} items")
if all_search_results['flights']:
    print(f"  Sample: {list(all_search_results['flights'][0].keys())[:3]}")
print(f"Hotels: {len(all_search_results['hotels'])} items")
if all_search_results['hotels']:
    print(f"  Sample: {list(all_search_results['hotels'][0].keys())[:3]}")
print(f"Activities: {len(all_search_results['activities'])} items")
if all_search_results['activities']:
    print(f"  Sample: {list(all_search_results['activities'][0].keys())[:3]}")


In [ ]:
# 4. Verifier: check draft against constraints
print("\n[Verifier] Checking itinerary against constraints...")
full_draft = "\n\n".join(draft_components)

# Build verifier input with specialist contexts
verifier_input = {
    "draft_text": full_draft,
    "constraints_json": constraints_str,
    "task_id": state.task_id,
    "search_results": all_search_results,
    "specialist_contexts": specialist_contexts,
}

try:
    verification = verify_and_format_itinerary(
        verifier_input["draft_text"],
        verifier_input["constraints_json"],
        task_id=verifier_input["task_id"],
        search_results=verifier_input["search_results"],
    )

    # Display verification result
    print("\n" + "=" * 60)
    print("VERIFICATION RESULT")
    print("=" * 60)
    print(f"Valid: {verification.get('is_valid')}")
    
    if verification.get("issues"):
        print(f"Issues: {verification.get('issues')}")
    
    if verification.get("is_valid"):
        # ✨ Display complete itinerary direct from result
        print("\n" + "=" * 60)
        print("COMPLETE TRAVEL ITINERARY (from verifier)")
        print("=" * 60)
        
        itinerary = verification.get("itinerary")
        if itinerary:
            if itinerary.get("flights"):
                print("\n✈️  FLIGHTS:")
                if itinerary["flights"].get("outbound"):
                    print(f"  Outbound: {itinerary['flights']['outbound']}")
                if itinerary["flights"].get("return"):
                    print(f"  Return: {itinerary['flights']['return']}")
            
            if itinerary.get("hotels"):
                print("\n🏨 HOTELS:")
                for key, value in itinerary["hotels"].items():
                    print(f"  {key}: {value}")
            
            if itinerary.get("activities"):
                print("\n🎯 ACTIVITIES:")
                for i, activity in enumerate(itinerary["activities"], 1):
                    print(f"  {i}. {activity}")
            
            print(f"\n💰 TOTAL COST: ${itinerary.get('estimated_cost', 0):,.2f}")
        
        final_response = verification.get(
            "final_message_to_user", "Here is your plan!"
        )
        print("\n✅ Nomad (Final Recommendation):\n")
        print(final_response)
        state.messages.append({"role": "assistant", "content": final_response})
    else:
        issues = verification.get("issues", [])
        print(
            f"\n❌ Nomad: I built a plan, but it violates some constraints:\n{issues}"
        )
        print("I will need to revise. Let me know how you'd like to adjust.")

except Exception as e:
    print(f"Error during verification: {e}")
    import traceback
    traceback.print_exc()



[Verifier] Checking itinerary against constraints...
[Saved] Verification result -> c:\GIT\aiagent\nomad\src\agents\..\verification_results\task_123_verification.json

VERIFICATION RESULT
Valid: True

Blueprint: {'flights': {'outbound_ref': 'flight_0', 'return_ref': 'flight_0'}, 'hotels': {'hotel_ref': 'hotel_1', 'check_in': '2026-05-05', 'check_out': '2026-05-12', 'nights': 7}, 'activities': [], 'estimated_cost': 589}

COMPLETE TRAVEL ITINERARY

📍 FLIGHTS:

🏨 HOTELS:

🎯 ACTIVITIES:

💰 TOTAL COST: $589.00

✅ Nomad (Final Recommendation):

Itinerary valid - 7-day Seattle trip with Southwest flights and Green Tortoise Hostel for $589 total


In [11]:
verification.get("complete_itinerary")

{'flights': {},
 'hotels': {},
 'activities': [],
 'cost_breakdown': {},
 'estimated_cost': 589}

In [ ]:

# 5. Format and display complete itinerary using formatter (if valid)
if verification.get("is_valid"):
    print("\n" + "=" * 70)
    print("FORMATTED OUTPUT")
    print("=" * 70)
    formatted = format_complete_itinerary(verification)
    print(formatted)

# Debug: Check itinerary structure
print("\n" + "=" * 70)
print("DEBUG: Itinerary Structure (Direct Access)")
print("=" * 70)
itinerary = verification.get("itinerary")
if itinerary:
    print(f"✈️  Flights: {bool(itinerary.get('flights'))}")
    if itinerary.get("flights"):
        print(f"    - Outbound populated: {bool(itinerary['flights'].get('outbound'))}")
        print(f"    - Return populated: {bool(itinerary['flights'].get('return'))}")
        if itinerary['flights'].get("outbound"):
            print(f"      Outbound keys: {list(itinerary['flights']['outbound'].keys())[:3]}")
    
    print(f"🏨 Hotels: {bool(itinerary.get('hotels'))}")
    if itinerary.get("hotels"):
        print(f"    Hotel keys: {list(itinerary['hotels'].keys())[:5]}")
    
    print(f"🎯 Activities: {len(itinerary.get('activities', []))} items")
    if itinerary.get("activities"):
        for i, act in enumerate(itinerary['activities'][:2]):
            print(f"    Activity {i}: {list(act.keys())}")
    
    print(f"💰 Estimated Cost: ${itinerary.get('estimated_cost', 0)}")
else:
    print("⚠️  No itinerary in verification result!")
    print(f"Verification keys: {list(verification.keys())}")



FORMATTED COMPLETE ITINERARY
✈️  COMPLETE TRAVEL ITINERARY

✅ STATUS: APPROVED

----------------------------------------------------------------------
💰 COST SUMMARY
----------------------------------------------------------------------
  Total Estimated Cost: $589.00

📝 Itinerary valid - 7-day Seattle trip with Southwest flights and Green Tortoise Hostel for $589 total


In [ ]:

def main():
    print("Welcome to Nomad Travel Agent!")
    print("Type 'quit' or 'exit' to stop.")
    print("-----------------------------------")

    state = TravelState()

    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ["quit", "exit"]:
            break

        # 1. Orchestrator: classify intent, extract constraints, decide delegation
        print("\n[Orchestrator] Analyzing input and updating Constraint Layer...")
        try:
            analysis = analyze_user_input(user_input, state)
            state = update_state_from_analysis(state, analysis)

            print(f"[Orchestrator] Intent: {analysis.get('intent')}")
            print(f"[Orchestrator] Delegation: {analysis.get('delegation')}")

            delegation = analysis.get("delegation", "none")

        except Exception as e:
            print(f"Error during orchestration: {e}")
            continue

        # 2. Add to message history
        state.messages.append({"role": "user", "content": user_input})

        # 3. Delegation
        if delegation == "none":
            response = analysis.get(
                "response_to_user",
                "I'm still missing some info to start planning.",
            )
            print(f"\nNomad: {response}")
            state.messages.append({"role": "assistant", "content": response})
            continue

        # If we reach here, we need to run specialists
        draft_components = []
        constraints_str = state.constraints.model_dump_json(indent=2)

        if delegation in ["logistics", "both"]:
            print("\n[Specialist - Logistics] Searching for flights & hotels...")
            try:
                logistics_draft = run_logistics_specialist(constraints_str)
                draft_components.append("--- LOGISTICS ---\n" + logistics_draft)
            except Exception as e:
                print(f"[Specialist Error] {e}")

        if delegation in ["activities", "both"]:
            print(
                "\n[Specialist - Activities] Searching for restaurants & things to do..."
            )
            try:
                activities_draft = run_activities_specialist(constraints_str)
                draft_components.append("--- ACTIVITIES ---\n" + activities_draft)
            except Exception as e:
                print(f"[Specialist Error] {e}")

        # 4. Verifier: check draft against constraints
        print("\n[Verifier] Checking itinerary against constraints...")
        full_draft = "\n\n".join(draft_components)

        try:
            verification = verify_and_format_itinerary(full_draft, constraints_str)

            if verification.get("is_valid"):
                final_response = verification.get(
                    "final_message_to_user", "Here is your plan!"
                )
                print("\nNomad:\n")
                print(final_response)
                state.messages.append({"role": "assistant", "content": final_response})
            else:
                issues = verification.get("issues", [])
                print(
                    f"\nNomad: I built a plan, but it violates some constraints:\n{issues}"
                )
                print("I will need to revise. Let me know how you'd like to adjust.")

        except Exception as e:
            print(f"Error during verification: {e}")


if __name__ == "__main__":
    main()